In [ ]:
from flow import Flow, FlowConfig
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
%load_ext autoreload
%autoreload 2

In [ ]:
config = FlowConfig()
flow = Flow(config=config)
config

In [ ]:
optim = torch.optim.Adam(flow.parameters(), lr=1e-3)

In [ ]:
dest_dist = torch.distributions.Normal(4, 1)
plt.hist(dest_dist.sample((1000,)),bins=100)
plt.show()

In [ ]:
epoches = 10_000
for epoch in range(epoches):
    x0 = dest_dist.sample((config.batch_size, config.input_dim))
    loss = flow.fm_training_step(x0)
    optim.zero_grad()
    loss.backward()
    optim.step()
    print(f"[Epoch: {epoch:07d}/{epoches:07d}], [Loss: {loss.item():3f}]")

In [ ]:
data, _ = flow.sample_actions(sample_num=5000)
plt.hist(data.flatten(), bins=100)
plt.show()

In [ ]:
@dataclass
class RewardModel:
    mu: float
    sigma: float

    def __call__(self, x):
        return torch.exp(-(x - self.mu)**2 / (2 * self.sigma * self.sigma))

In [ ]:
reward_model = RewardModel(-3, 1)
x = torch.linspace(-4, 10, 100)
plt.plot(x, reward_model(x) * 170, c='r')
data, _ = flow.sample_actions(sample_num=5000)
plt.hist(data.flatten(), bins=100)
plt.show()

In [ ]:
rl_epoches = 5000
for epoch in range(rl_epoches):
    _, bags = flow.sample_actions()
    last_loss: float = flow.rl_training_step(bags, reward_model, optim)
    print(f"[Epoch: {epoch: 04d}/{rl_epoches:04d}], [Loss: {last_loss:.8f}]")

x = torch.linspace(-10, 10, 100)
plt.plot(x, reward_model(x) * 170, c='r')
data, _ = flow.sample_actions(sample_num=5000)
plt.hist(data.flatten(), bins=100)
plt.show()